# Задача 3. K-th

`kth_largest(nums, k)` возвращает k-й элемент в порядке убывания. Счёт начинается с 1, одинаковые числа занимают отдельные позиции. Функция копирует вход, поэтому исходный массив остаётся прежним. При `k` вне диапазона `1…len(nums)` возникает `ValueError`.

## Как работает Quickselect

Искомый элемент в порядке убывания имел бы индекс `target = k - 1`. Берём случайный опорный элемент `pivot` внутри текущего диапазона `[left, right]` и за один проход делим диапазон на три области: больше `pivot`, равные ему, меньше него. Границы равной области после прохода — `[greater, smaller]`. Внутри остальных областей порядок элементов не важен.

- Если `target < greater`, продолжаем только в области больших элементов.
- Если `target > smaller`, продолжаем только в области меньших.
- Если индекс попал в равную область, возвращаем `pivot`: все её позиции имеют одно значение.

Во время разбиения `current` указывает на следующий ещё не разобранный элемент. Когда он меньше опорного, перемещаем его вправо. На его место приходит ещё не проверенное значение, поэтому `current` остаётся на месте. После разбиения область поиска строго уменьшается, так как равная область содержит хотя бы опорный элемент. Если остаётся одна позиция, она и есть искомая. Алгоритм не сортирует массив и не использует кучу.

## Реализация

In [1]:
"""Поиск k-го по величине элемента без сортировки массива."""

from collections.abc import Sequence
from random import Random, randrange
from typing import Optional


def kth_largest(nums: Sequence[int], k: int, *, rng: Optional[Random] = None) -> int:
    """Вернуть k-е наибольшее значение (k начинается с 1, повторы считаются).

    Входная последовательность не изменяется. Необязательный rng позволяет
    воспроизвести последовательность опорных элементов.
    """
    if isinstance(k, bool) or not isinstance(k, int):
        raise TypeError("k должно быть целым числом")
    if not 1 <= k <= len(nums):
        raise ValueError("k должно быть от 1 до длины массива")

    items = list(nums)
    target = k - 1
    left, right = 0, len(items) - 1
    choose_index = randrange if rng is None else rng.randrange

    while left < right:
        pivot = items[choose_index(left, right + 1)]
        greater, current, smaller = left, left, right

        # [left, greater) > pivot; [greater, current) == pivot;
        # (smaller, right] < pivot; [current, smaller] ещё не обработан.
        while current <= smaller:
            if items[current] > pivot:
                items[greater], items[current] = items[current], items[greater]
                greater += 1
                current += 1
            elif items[current] < pivot:
                items[current], items[smaller] = items[smaller], items[current]
                smaller -= 1
            else:
                current += 1

        if target < greater:
            right = greater - 1
        elif target > smaller:
            left = smaller + 1
        else:
            return pivot

    return items[left]

## Пример использования

In [2]:
print(kth_largest([3, 2, 1, 5, 6, 4], 2, rng=Random(1)))
print(kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4, rng=Random(1)))

5
4


## Пошаговый пример

Для `nums = [3, 2, 3, 1, 2, 4, 5, 5, 6]`, `k = 4` ищем индекс `3`. С `rng=Random(1)` получаем такие состояния (индексы с 0):

| Шаг | Диапазон | Опорное значение | Массив после разбиения | Область равных | Действие |
| --- | --- | --- | --- | --- | --- |
| 1 | `[0, 8]` | `3` | `[6, 5, 5, 4, 3, 3, 2, 1, 2]` | `[4, 5]` | `3 < 4`: ищем в `[0, 3]` |
| 2 | `[0, 3]` | `6` | `[6, 5, 4, 5, 3, 3, 2, 1, 2]` | `[0, 0]` | `3 > 0`: ищем в `[1, 3]` |
| 3 | `[1, 3]` | `4` | `[6, 5, 5, 4, 3, 3, 2, 1, 2]` | `[3, 3]` | Индекс попал в область равных: возвращаем `4` |

Две пятёрки занимают два места перед четвёркой, поэтому ответ на четвёртой позиции — `4`.

## Сложность

- **Время: ожидаемое `O(n)`, худшее `O(n²)`.** Разбиение стоит `O(m)` для диапазона длины `m`, и дальше берём только одну часть. С вероятностью не меньше примерно половины случайный опорный элемент попадает в среднюю половину рангов, тогда остаётся не больше примерно `3m/4` элементов. До такого уменьшения нужно в среднем постоянное число разбиений, поэтому суммарная ожидаемая работа линейна. Если опорный элемент каждый раз отсекает только одну позицию, сумма длин диапазонов квадратична.
- **Дополнительная память: `O(n)`** для копии входа. Само итеративное разбиение использует `O(1)` памяти и не создаёт стек рекурсии.

## Тесты

In [3]:
import unittest
from random import Random



class KthLargestTests(unittest.TestCase):
    def test_examples(self):
        cases = [
            ([3, 2, 1, 5, 6, 4], 2, 5),
            ([3, 2, 3, 1, 2, 4, 5, 5, 6], 4, 4),
        ]
        for nums, k, expected in cases:
            with self.subTest(nums=nums, k=k):
                self.assertEqual(kth_largest(nums, k, rng=Random(1)), expected)

    def test_singleton(self):
        self.assertEqual(kth_largest((7,), 1), 7)

    def test_boundary_ranks_and_negative_values(self):
        nums = [0, -7, 3, 1, -2]
        self.assertEqual(kth_largest(nums, 1), 3)
        self.assertEqual(kth_largest(nums, len(nums)), -7)

    def test_duplicates_occupy_separate_ranks(self):
        nums = [2, 5, 5, 1, 5, 4]
        expected = [5, 5, 5, 4, 2, 1]
        for k, value in enumerate(expected, start=1):
            with self.subTest(k=k):
                self.assertEqual(kth_largest(nums, k, rng=Random(4)), value)

    def test_all_values_equal(self):
        self.assertEqual(kth_largest([11] * 10_000, 9_999, rng=Random(2)), 11)

    def test_does_not_change_input(self):
        nums = [9, 1, 7, 3, 7, 2]
        before = nums.copy()
        self.assertEqual(kth_largest(nums, 3, rng=Random(10)), 7)
        self.assertEqual(nums, before)

    def test_rejects_out_of_range_rank(self):
        for nums, k in [([], 1), ([1], 0), ([1], -1), ([1], 2)]:
            with self.subTest(nums=nums, k=k):
                with self.assertRaises(ValueError):
                    kth_largest(nums, k)

    def test_rejects_non_integer_rank(self):
        for k in (1.0, "1", True):
            with self.subTest(k=k):
                with self.assertRaises(TypeError):
                    kth_largest([1], k)

    def test_matches_reference_on_generated_inputs(self):
        inputs = Random(271)
        pivots = Random(822)
        for size in range(1, 36):
            nums = [inputs.randint(-12, 12) for _ in range(size)]
            reference = sorted(nums, reverse=True)
            for k, value in enumerate(reference, start=1):
                with self.subTest(size=size, k=k):
                    self.assertEqual(kth_largest(nums, k, rng=pivots), value)

suite = unittest.TestSuite()
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(KthLargestTests))
report = unittest.TextTestRunner(verbosity=2).run(suite)
assert report.wasSuccessful(), 'Не все тесты прошли'

test_all_values_equal (__main__.KthLargestTests.test_all_values_equal) ... 

ok


test_boundary_ranks_and_negative_values (__main__.KthLargestTests.test_boundary_ranks_and_negative_values) ... 

ok


test_does_not_change_input (__main__.KthLargestTests.test_does_not_change_input) ... 

ok


... (9 outputs left out)

ok


test_singleton (__main__.KthLargestTests.test_singleton) ... 

ok


----------------------------------------------------------------------
Ran 9 tests in 0.009s

OK
